## Task: Generating tweets using n_grams

In [1]:
pip install emoji

   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/610.9 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/610.9 kB ? eta -:--:--
   ----------------- ---------------------- 262.1/610.9 kB ? eta -:--:--
   -------------------------------------- 610.9/610.9 kB 769.3 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Load the Dataset


In [6]:
import pandas as pd

# Load the CSV file with latin1 encoding to handle special characters safely
df = pd.read_csv('Random Tweets from Pakistan- Cleaned- Anonymous.csv', encoding='latin1')

# Extract the text column containing the tweets
tweets = df['full_text'].dropna().tolist()

print(f"Successfully loaded {len(tweets)} tweets!")

Successfully loaded 202151 tweets!


C:\Users\Ahmed\AppData\Local\Temp\ipykernel_20684\2051254600.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Random Tweets from Pakistan- Cleaned- Anonymous.csv', encoding='latin1')


In [7]:
import pandas as pd

# Load only the full_text column to skip warnings and save memory
df = pd.read_csv('Random Tweets from Pakistan- Cleaned- Anonymous.csv', encoding='latin1', usecols=['full_text'])

# Extract the text column containing the tweets
tweets = df['full_text'].dropna().tolist()

print(f"Successfully loaded {len(tweets)} tweets!")

Successfully loaded 202151 tweets!


# Pre-process the Tweets

In [8]:
import re
import emoji

def preprocess_tweet(text):
    if not isinstance(text, str):
        return ""
    # Remove RT (Retweet indicator)
    text = re.sub(r'\bRT\b', '', text)
    # Remove websites / URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove mentions (@username)
    text = re.sub(r'@\w+', '', text)
    # Remove hashtags (#hashtag)
    text = re.sub(r'#\w+', '', text)
    # Remove emojis using the emoji library
    text = emoji.replace_emoji(text, replace='')
    # Clean up extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Using a sample of 10,000 tweets for quick execution (you can remove '[:10000]' to use all)
sample_tweets = tweets[:10000]
cleaned_tweets = [preprocess_tweet(t) for t in sample_tweets if preprocess_tweet(t) != '']

print(f"Cleaned {len(cleaned_tweets)} tweets successfully!")
for t in cleaned_tweets[:3]:
    print("-", t)

Cleaned 9482 tweets successfully!
- ØªÛØ±Ø§ ÙÛÚØ± Ù ÛØ±Ø§ ÙÛÚØ± ÙÙØ§Ø² Ø´Ø±ÛÙ ÙÙØ§Ø² Ø´Ø±ÛÙâ¤ï¸ ÚÛØ§ Ø¢Ù¾ÚØ§ Ø¨Ú¾Û Ú¾ÛØØØ
- Happy birthday to my brother n boss , May you have many many moreððð, you are Gem ð
- â¤ï¸â¤ï¸


# Tokenization and Building the Bigram MLE Model

In [10]:
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

# Tokenize cleaned tweets into lowercased words
tokenized_tweets = [word_tokenize(tweet.lower()) for tweet in cleaned_tweets]

# Build padded everygram pipeline for bigrams (n=2)
train_data, vocab = padded_everygram_pipeline(2, tokenized_tweets)

# Convert vocab to a list so we can inspect it and get its length
vocab_list = list(vocab)

# Initialize and train the MLE bigram model (Note: pass vocab_list since vocab was already consumed)
lm = MLE(2)
lm.fit(train_data, vocab_list)

print(f"Model trained successfully! Vocabulary size: {len(vocab_list)}")

Model trained successfully! Vocabulary size: 139495


# Evaluate Probability and Perplexity

In [11]:
# 1. Calculate the probability P('is' | 'pakistan')
prob_pakistan_is = lm.score('is', ['pakistan'])
print(f"Probability P('is' | 'pakistan'): {prob_pakistan_is}")

# 2. Calculate perplexity for a test bigram sequence
test_sequence = [('pakistan', 'is')]
perplexity_val = lm.perplexity(test_sequence)
print(f"Perplexity for test sequence: {perplexity_val}")

Probability P('is' | 'pakistan'): 0.0364963503649635
Perplexity for test sequence: 27.400000000000006
